# Painter Classification with FastAI

This notebook presents a complete training and evaluation pipeline for fine-tuning a ResNet50 model on a 50-class painter classification task using the [Best Artworks of All Time](https://www.kaggle.com/datasets/ikarus777/best-artworks-of-all-time) Kaggle dataset.<br>Credit to [Icaro](https://www.kaggle.com/ikarus777) for sharing this excellent dataset!<br>The dataset contains 8,446 paintings from 50 artists, with class imbalance (ranging from ~25 to 800+ paintings per artist).

This notebook uses the FastAI library.<br>
I also created a version using the PyTorch library here: **[PyTorch version](https://www.kaggle.com/code/amarietiberiu/pytorch-painter-classification)**.<br>
For exploratory data analysis on this dataset, check out this notebook: **[EDA](https://www.kaggle.com/code/amarietiberiu/eda-painter-classification)**.


We achieve **~86% accuracy** and **~0.82 macro-average F1** on the validation set.

In [ ]:
!pip install -Uqq fastai
!pip uninstall -y fastprogress
!pip install "fastprogress==1.0.3"

In [ ]:
from fastai.vision.all import *

In [ ]:
# Ensure we have the data available
import kagglehub

path = '/kaggle/input/datasets/ikarus777/best-artworks-of-all-time'
if not os.environ.get('KAGGLE_KERNEL_RUN_TYPE', ''):
    path = kagglehub.dataset_download("ikarus777/best-artworks-of-all-time")
data_path = Path(path)/'images'/'images'
print(f"Path exists: {data_path.exists()}") 

In [ ]:
def get_class_from_filename(x: str) -> str:
    """Helper function for extracting the class from the image file name"""
    return x.rsplit("_", 1)[0]

def get_unique_images(x: Path) -> list[Path]:
    """Helper function for collecting all images from the training folder"""
    return [path for path in x.rglob("*.jpg")
            if get_class_from_filename(path.name) != 'Albrecht_Du╠êrer'] # Exclude duplicate folder

In [ ]:
# Exclude duplicate folder 'Albrecht_Du╠êrer' (mojibake variant of 'Albrecht Dürer')
labels = [get_class_from_filename(path.name) for path in get_unique_images(data_path)]

dls = ImageDataLoaders.from_name_func(
    data_path,
    get_image_files(data_path, recurse=True, folders=labels),
    get_class_from_filename,
    valid_pct=0.15,
    item_tfms=[Resize(448)],
    bs=32
)

In [ ]:
dls.show_batch()

# Training

We will use the powerful transfer learning technique to fine-tune the ResNet50 model, pre-trained on ImageNet, with our dataset. The FastAI library has very good default settings for fine-tuning, so we will keep the custom parameters to a minimum.

In [ ]:
learner = vision_learner(dls, resnet50, metrics=[accuracy, F1Score(average='macro')])
learner.fine_tune(15) # Fine tune the model for 15 epochs

# Results

The model reaches ~86% accuracy with minimal code thanks to FastAI's transfer learning defaults.

In [ ]:
learner.show_results()

In [ ]:
Interpretation.from_learner(learner).plot_top_losses(9, figsize=(15, 10))

#### That's it!
#### Please share your thoughts and feedback in the comments. If you liked the notebook please "upvote" 🔝
#### Check out [https://github.com/Tiberiw](https://github.com/Tiberiw) for other projects I am working on :)